In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import pandas as pd
from src.data_loader import load_datasets, get_train_test_split

print('Loading datasets...')
lob_df, articles_df = load_datasets()
train_df, test_df = get_train_test_split()

print(f'Total articles: {len(articles_df)}')
print(f'Train set: {len(train_df)}')
print(f'Test set:  {len(test_df)}')
print(f'Unique LOBs in test: {test_df["lob_code_str"].nunique()}')
print('\nClass distribution (top 10 LOBs in test):')
print(test_df['lob_code_str'].value_counts().head(10))

Loading datasets...
Total articles: 20499
Train set: 17426
Test set:  3073
Unique LOBs in test: 99

Class distribution (top 10 LOBs in test):
lob_code_str
1002     301
3004     272
4009     232
2002     197
17001    145
3014     120
24001     88
6002      78
2001      74
1001      74
Name: count, dtype: int64


In [2]:
def compute_metrics(results_df: pd.DataFrame) -> dict:
    """
    results_df columns:
      article_code, true_lob, pred_lob_1, pred_lobs_top3 (list),
      pred_inventory_1, true_inventory, confidence_1, correct_top1, correct_top3
    """
    top1 = results_df['correct_top1'].mean()
    top3 = results_df['correct_top3'].mean()

    inventory_mask = results_df['correct_top1']
    inv_acc = (
        results_df.loc[inventory_mask, 'pred_inventory_1'] ==
        results_df.loc[inventory_mask, 'true_inventory']
    ).mean() if inventory_mask.sum() > 0 else float('nan')

    mean_conf_correct = results_df.loc[results_df['correct_top1'], 'confidence_1'].mean()
    mean_conf_wrong = results_df.loc[~results_df['correct_top1'], 'confidence_1'].mean()

    return {
        'top1_accuracy': round(top1, 4),
        'top3_accuracy': round(top3, 4),
        'inventory_accuracy_when_top1_correct': round(inv_acc, 4) if inv_acc == inv_acc else None,
        'mean_confidence_correct': round(mean_conf_correct, 4) if mean_conf_correct == mean_conf_correct else None,
        'mean_confidence_incorrect': round(mean_conf_wrong, 4) if mean_conf_wrong == mean_conf_wrong else None,
    }

In [3]:
from tqdm import tqdm
from src.graph import classify_article
from src.vectorstore_setup import initialize_vectorstore
from src.data_loader import load_datasets, get_train_test_split

# Initialize vectorstore (skips if already built)
lob_df, _ = load_datasets()
train_df, test_df = get_train_test_split()
print(len(train_df), len(test_df))
initialize_vectorstore(train_df, lob_df)

SAMPLE_SIZE = 100
SAVE_EVERY = 10
RESULTS_PATH = 'eval_results.csv'

sample = test_df.sample(n=min(SAMPLE_SIZE, len(test_df)), random_state=42)

records = []
for i, (_, row) in enumerate(tqdm(sample.iterrows(), total=len(sample))):
    code = row['codice_articolo']
    true_lob = row['lob_code_str']
    true_inv = row['inventario']

    try:
        result = classify_article(code)
        print(result)
        suggestions = result.get('suggestions', [])

        pred_lob_1 = suggestions[0]['lob_code'] if suggestions else None
        pred_inv_1 = suggestions[0]['inventory'] if suggestions else None
        confidence_1 = suggestions[0]['confidence'] if suggestions else 0.0
        pred_lobs_top3 = [s['lob_code'] for s in suggestions[:3]]
    except Exception as exc:
        print(exc)
        pred_lob_1, pred_inv_1, confidence_1, pred_lobs_top3 = None, None, 0.0, []

    records.append({
        'article_code': code,
        'true_lob': true_lob,
        'true_inventory': true_inv,
        'pred_lob_1': pred_lob_1,
        'pred_inventory_1': pred_inv_1,
        'confidence_1': confidence_1,
        'pred_lobs_top3': pred_lobs_top3,
        'correct_top1': pred_lob_1 == true_lob,
        'correct_top3': true_lob in pred_lobs_top3,
    })

    if (i + 1) % SAVE_EVERY == 0:
        pd.DataFrame(records).to_csv(RESULTS_PATH, index=False)
        print(f'Saved {i+1} results to {RESULTS_PATH}')

results_df = pd.DataFrame(records)
results_df.to_csv(RESULTS_PATH, index=False)
print(f'Evaluation complete. {len(results_df)} articles evaluated.')

17426 3073


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore already populated — skipping embedding.


  1%|          | 1/100 [01:46<2:55:30, 106.36s/it]


{'article_code': 'OS01137233', 'article_description': 'Apex One on-prem includes Mac VDI iDLP iVP iAC and Apex Central New Normal 101-250 User License12', 'existing_lob': None, 'existing_inventory': 'Inventario', 'web_enrichment': 'Software license, consumabile, prodotto autonomo.  \nCiclo di vita 12 mesi, categoria consumabile (spesa diretta).', 'suggestions': [{'rank': 1, 'lob_code': '6012', 'lob_name': 'RINNOVO SOFTWARE TREND MICRO', 'inventory': 'Non in inventario', 'explanation': "L'articolo rappresenta una licenza software consumabile con ciclo di vita 12 mesi, conforme a prodotti simili classificati in 'Non in inventario'. La descrizione 'consumabile' e 'spesa diretta' conferma il trattamento non in inventario, pur se il codice LOB indica un rinnovo, poiché il contesto storico mostra licenze simili con lo stesso codice.", 'confidence': 0.975}, {'rank': 2, 'lob_code': '6026', 'lob_name': 'TREND MICRO ADVANCED SECURITY SUITE', 'inventory': 'Non in inventario', 'explanation': "La 

  1%|          | 1/100 [01:57<3:13:15, 117.12s/it]


KeyboardInterrupt: 

In [ ]:
metrics = compute_metrics(results_df)
summary = pd.DataFrame([metrics]).T
summary.columns = ['value']
print('=== EVALUATION SUMMARY ===')
print(summary.to_string())

=== EVALUATION SUMMARY ===
                                     value
top1_accuracy                          0.0
top3_accuracy                          0.0
inventory_accuracy_when_top1_correct  None
mean_confidence_correct               None
mean_confidence_incorrect              0.0


In [ ]:
import matplotlib.pyplot as plt

# Top 10 confusion pairs
error_df = results_df[~results_df['correct_top1']].copy()
if len(error_df) > 0:
    error_df['pair'] = error_df['true_lob'] + ' → ' + error_df['pred_lob_1'].fillna('None')
    top_confusions = error_df['pair'].value_counts().head(10)
    print('Top 10 confused LOB pairs (true → predicted):')
    print(top_confusions.to_string())
else:
    print('No errors to analyze.')

# Example incorrect classifications
print('\n--- Example incorrect classifications ---')
for _, row in error_df.head(5).iterrows():
    print(f"Article: {row['article_code']}")
    print(f"  True LOB: {row['true_lob']}, Predicted: {row['pred_lob_1']}, Confidence: {row['confidence_1']:.3f}")

# Confidence distribution histogram
fig, ax = plt.subplots(figsize=(8, 4))
correct_conf = results_df.loc[results_df['correct_top1'], 'confidence_1']
wrong_conf = results_df.loc[~results_df['correct_top1'], 'confidence_1']
ax.hist(correct_conf, bins=20, alpha=0.6, label='Correct (Top-1)', color='green')
ax.hist(wrong_conf, bins=20, alpha=0.6, label='Incorrect (Top-1)', color='red')
ax.set_xlabel('Confidence score')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution: Correct vs Incorrect Predictions')
ax.legend()
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=100)
plt.show()
print('Saved: confidence_distribution.png')

In [ ]:
lob_stats = (
    results_df.groupby('true_lob')
    .agg(
        n_samples=('correct_top1', 'count'),
        top1_acc=('correct_top1', 'mean'),
        top3_acc=('correct_top3', 'mean'),
    )
    .reset_index()
    .sort_values('top1_acc')
)

# Join LOB names
lob_names = lob_df[['LOB Code', 'Name']].rename(columns={'LOB Code': 'true_lob'})
lob_stats = lob_stats.merge(lob_names, on='true_lob', how='left')

print('Per-LOB accuracy (worst first):')
print(lob_stats[['true_lob', 'Name', 'n_samples', 'top1_acc', 'top3_acc']].to_string(index=False))

In [ ]:
from src.state import AgentState
from src.nodes.db_lookup import db_lookup_node
from src.nodes.rag_classification import rag_classification_node
from src.data_loader import get_train_test_split


def classify_article_no_web(article_code: str) -> dict:
    """Classification pipeline without web enrichment (Node 2 skipped)."""
    train_df, _ = get_train_test_split()
    from src.data_loader import get_article_info
    article_info = get_article_info(article_code, train_df)

    if article_info is None:
        return {'article_code': article_code, 'suggestions': [], 'error': 'not found'}

    state: AgentState = {
        'article_code': article_code,
        'article_info': article_info,
        'web_enrichment': '',
        'retrieval_results': [],
        'classification': [],
        'error': None,
    }

    rag_result = rag_classification_node(state)
    return {
        'article_code': article_code,
        'suggestions': rag_result.get('classification', []),
        'error': rag_result.get('error'),
    }


# Run no-web evaluation on same sample
records_no_web = []
for i, (_, row) in enumerate(tqdm(sample.iterrows(), total=len(sample))):
    code = row['codice_articolo']
    true_lob = row['lob_code_str']
    true_inv = row['inventario']

    try:
        result = classify_article_no_web(code)
        suggestions = result.get('suggestions', [])
        pred_lob_1 = suggestions[0]['lob_code'] if suggestions else None
        pred_inv_1 = suggestions[0]['inventory'] if suggestions else None
        confidence_1 = suggestions[0]['confidence'] if suggestions else 0.0
        pred_lobs_top3 = [s['lob_code'] for s in suggestions[:3]]
    except Exception:
        pred_lob_1, pred_inv_1, confidence_1, pred_lobs_top3 = None, None, 0.0, []

    records_no_web.append({
        'article_code': code,
        'true_lob': true_lob,
        'true_inventory': true_inv,
        'pred_lob_1': pred_lob_1,
        'pred_inventory_1': pred_inv_1,
        'confidence_1': confidence_1,
        'pred_lobs_top3': pred_lobs_top3,
        'correct_top1': pred_lob_1 == true_lob,
        'correct_top3': true_lob in pred_lobs_top3,
    })

results_no_web_df = pd.DataFrame(records_no_web)
metrics_no_web = compute_metrics(results_no_web_df)

print('=== WITH WEB ENRICHMENT ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

print('\n=== WITHOUT WEB ENRICHMENT ===')
for k, v in metrics_no_web.items():
    print(f'  {k}: {v}')